# Notebook 03 — Model Training & Evaluation

**Project:** BudgetWise — Expense Tracker & Budget Planner with ML  
**Purpose:** Train, evaluate, and serialise three ML models:
1. **ExpensePredictor** — forecast next month's spending per category
2. **CategoryClassifier** — auto-classify transactions from description
3. **AnomalyDetector** — save per-category statistical baselines

Trained models are saved to `../saved_models/` and loaded by the Flask backend.

---

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report

MODELS_DIR = '../saved_models/'
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Category Classifier

In [ ]:
clf_data = pd.read_csv('../data/processed/classification_data.csv')
X = clf_data['clean_description']
y = clf_data['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf_pipeline = Pipeline([
    ('tfidf',  TfidfVectorizer(max_features=1000, ngram_range=(1, 2))),
    ('clf',    LogisticRegression(max_iter=500, C=1.0)),
])

clf_pipeline.fit(X_train, y_train)
y_pred = clf_pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

# Save model
joblib.dump(clf_pipeline, os.path.join(MODELS_DIR, 'category_classifier.joblib'))
print('\n✓ Category classifier saved.')

## 2. Expense Forecasting Model

In [ ]:
forecast_data = pd.read_csv('../data/processed/forecasting_features.csv')

FEATURE_COLS = ['year', 'month', 'lag_1', 'lag_2', 'rolling_mean_3']
TARGET_COL   = 'monthly_total'

# Encode category as a numeric feature
le = LabelEncoder()
forecast_data['category_enc'] = le.fit_transform(forecast_data['category'])
FEATURE_COLS.append('category_enc')

X = forecast_data[FEATURE_COLS]
y = forecast_data[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Absolute Error: ₦{mae:,.0f}')

# Bundle model + label encoder so we can inverse-transform category names
forecast_bundle = {'model': regressor, 'label_encoder': le, 'feature_cols': FEATURE_COLS}
joblib.dump(forecast_bundle, os.path.join(MODELS_DIR, 'expense_predictor.joblib'))
print('\n✓ Expense predictor saved.')

## 3. Anomaly Detection Baselines

In [ ]:
baselines = pd.read_csv('../data/processed/anomaly_baselines.csv', index_col=0)

# Convert to the dict format expected by AnomalyDetector
stats_dict = {
    row.Index: {'mean': row.mean_amount, 'std': row.std_amount}
    for row in baselines.itertuples()
}

joblib.dump(stats_dict, os.path.join(MODELS_DIR, 'anomaly_detector.joblib'))
print('Anomaly baselines saved:')
for cat, s in stats_dict.items():
    print(f'  {cat}: mean=₦{s["mean"]:,.0f}, std=₦{s["std"]:,.0f}')
print('\n✓ Anomaly detector saved.')

## 4. Summary

All three models have been saved to `../saved_models/`.  
The Flask backend will automatically load them when the server starts.

**Next steps:**
- Collect more real transaction data to improve model accuracy
- Experiment with XGBoost or LSTM for forecasting
- Add cross-validation and hyperparameter tuning
- Write unit tests for each ML class in `tests/backend/`